In [ ]:
import pandas as pd

val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/validation.tsv", sep="\t")
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/test.tsv", sep="\t")
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TFG/train.tsv", sep="\t")
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(149043, 124)
(31938, 124)
(31938, 124)


In [ ]:
import json
import numpy as np
train_df["fingerprint"] = train_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)
val_df["fingerprint"] = val_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)
test_df["fingerprint"] = test_df["fingerprint"].apply(
    lambda x: np.array(json.loads(x), dtype=np.float32)
)

In [ ]:
import ast
import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


# =========================================================
# 1. Función para convertir fingerprint a lista de bits
# =========================================================

def parse_fingerprint(fp):
    """
    Convierte la columna fingerprint a lista de números.

    Soporta:
    - listas reales de Python
    - strings del tipo "[0, 1, 0, ...]"
    - arrays / listas / tuplas
    """

    if isinstance(fp, list):
        return fp

    if isinstance(fp, tuple):
        return list(fp)

    if isinstance(fp, np.ndarray):
        return fp.tolist()

    if isinstance(fp, str):
        fp = fp.strip()

        try:
            parsed = ast.literal_eval(fp)

            if isinstance(parsed, (list, tuple, np.ndarray)):
                return list(parsed)

        except Exception:
            pass

    raise ValueError(f"No se pudo parsear fingerprint: {fp}")


# =========================================================
# 2. Función para expandir fingerprint
# =========================================================

def expand_fingerprint_column(df, fingerprint_col="fingerprint"):
    """
    Expande la columna fingerprint a columnas:
    fp_0, fp_1, ..., fp_n
    """

    fp_lists = df[fingerprint_col].apply(parse_fingerprint)

    lengths = fp_lists.apply(len)

    if lengths.nunique() != 1:
        raise ValueError(
            f"Los fingerprints no tienen todos la misma longitud. "
            f"Longitudes encontradas: {sorted(lengths.unique())}"
        )

    fp_len = lengths.iloc[0]

    fp_matrix = np.vstack(fp_lists.values).astype(np.float32)

    fp_df = pd.DataFrame(
        fp_matrix,
        columns=[f"fp_{i}" for i in range(fp_len)],
        index=df.index
    )

    return fp_df


# =========================================================
# 3. Función auxiliar para convertir a tensores PyTorch
# =========================================================

def to_torch_tensor(x, dtype=torch.float32, device=None):
    """
    Convierte numpy array / DataFrame / Series a tensor PyTorch.
    """

    if isinstance(x, pd.DataFrame) or isinstance(x, pd.Series):
        x = x.to_numpy()

    x = np.asarray(x)

    tensor = torch.tensor(x, dtype=dtype)

    if device is not None:
        tensor = tensor.to(device)

    return tensor


# =========================================================
# 4. Función para invertir escalado del target
# =========================================================

def inverse_transform_target(y_scaled, y_scaler):
    """
    Invierte la transformación del target.

    Acepta:
    - numpy array
    - pandas Series/DataFrame
    - tensor de PyTorch

    Devuelve:
    - numpy array 1D en escala original
    """

    if isinstance(y_scaled, torch.Tensor):
        y_scaled = y_scaled.detach().cpu().numpy()

    if isinstance(y_scaled, pd.Series):
        y_scaled = y_scaled.to_numpy()

    if isinstance(y_scaled, pd.DataFrame):
        y_scaled = y_scaled.to_numpy()

    y_scaled = np.asarray(y_scaled, dtype=np.float32).reshape(-1, 1)

    y_original = y_scaler.inverse_transform(y_scaled).reshape(-1)

    return y_original


# =========================================================
# 5. Función para separar X e y y expandir fingerprints
# =========================================================

def build_X_y(
    df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    fingerprint_col="fingerprint"
):
    """
    A partir de un dataframe, separa X e y:

    - convierte target a numérico
    - elimina filas sin target válido
    - expande fingerprint
    - elimina columnas de id, target y fingerprint original
    - concatena variables normales + fingerprint expandido
    """

    df = df.copy()

    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col]).copy()

    # Expandir fingerprint
    fp_df = expand_fingerprint_column(df, fingerprint_col=fingerprint_col)

    # Columnas que no deben entrar como input
    drop_cols = [target_col, fingerprint_col]

    for col in id_cols:
        if col in df.columns:
            drop_cols.append(col)

    X_base = df.drop(columns=drop_cols, errors="ignore")
    y = df[target_col].copy()

    # Añadir fingerprints expandidos
    X = pd.concat([X_base, fp_df], axis=1)

    return X, y


# =========================================================
# 6. Función principal de preprocesamiento
# =========================================================

def preprocess_rt_splits(
    train_df,
    val_df,
    test_df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    categorical_cols=("column.name", "column.usp.code"),
    fingerprint_col="fingerprint",
    scale_target=True,
    device=None
):
    """
    Preprocesa train, validation y test ya separados.

    Importante:
    - El preprocessor se ajusta SOLO con train.
    - Validation y test solo se transforman.
    - El y_scaler también se ajusta SOLO con y_train.
    """

    # =====================================================
    # A. Construir X e y para cada subset
    # =====================================================

    X_train, y_train = build_X_y(
        train_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    X_val, y_val = build_X_y(
        val_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    X_test, y_test = build_X_y(
        test_df,
        target_col=target_col,
        id_cols=id_cols,
        fingerprint_col=fingerprint_col
    )

    # =====================================================
    # B. Definir columnas categóricas, numéricas y fingerprint
    # =====================================================

    categorical_cols = [c for c in categorical_cols if c in X_train.columns]

    fp_cols = [c for c in X_train.columns if c.startswith("fp_")]

    numeric_cols = [
        c for c in X_train.columns
        if c not in categorical_cols and c not in fp_cols
    ]

    # =====================================================
    # C. Pipelines de preprocesado
    # =====================================================

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    fingerprint_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_pipeline, categorical_cols),
            ("num", numeric_pipeline, numeric_cols),
            ("fp", fingerprint_pipeline, fp_cols),
        ],
        remainder="drop"
    )

    # =====================================================
    # D. Ajustar SOLO con train
    # =====================================================

    X_train_processed = preprocessor.fit_transform(X_train)

    X_val_processed = preprocessor.transform(X_val)
    X_test_processed = preprocessor.transform(X_test)

    # =====================================================
    # E. Reconstruir DataFrames procesados
    # =====================================================

    feature_names = preprocessor.get_feature_names_out()

    X_train_processed = pd.DataFrame(
        X_train_processed,
        columns=feature_names,
        index=X_train.index
    ).astype(np.float32)

    X_val_processed = pd.DataFrame(
        X_val_processed,
        columns=feature_names,
        index=X_val.index
    ).astype(np.float32)

    X_test_processed = pd.DataFrame(
        X_test_processed,
        columns=feature_names,
        index=X_test.index
    ).astype(np.float32)

    # =====================================================
    # F. Guardar y original antes de escalar
    # =====================================================

    y_train_original = y_train.astype(np.float32).copy()
    y_val_original = y_val.astype(np.float32).copy()
    y_test_original = y_test.astype(np.float32).copy()

    # =====================================================
    # G. Escalado opcional del target
    # =====================================================

    y_scaler = None

    if scale_target:
        y_scaler = StandardScaler()

        y_train_scaled = y_scaler.fit_transform(
            y_train.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_val_scaled = y_scaler.transform(
            y_val.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_test_scaled = y_scaler.transform(
            y_test.to_numpy().reshape(-1, 1)
        ).reshape(-1).astype(np.float32)

        y_train_processed = pd.Series(
            y_train_scaled,
            index=y_train.index,
            name=target_col
        )

        y_val_processed = pd.Series(
            y_val_scaled,
            index=y_val.index,
            name=target_col
        )

        y_test_processed = pd.Series(
            y_test_scaled,
            index=y_test.index,
            name=target_col
        )

    else:
        y_train_processed = y_train.astype(np.float32)
        y_val_processed = y_val.astype(np.float32)
        y_test_processed = y_test.astype(np.float32)

    # =====================================================
    # H. Conversión a tensores PyTorch
    # =====================================================

    X_train_tensor = to_torch_tensor(
        X_train_processed,
        dtype=torch.float32,
        device=device
    )

    X_val_tensor = to_torch_tensor(
        X_val_processed,
        dtype=torch.float32,
        device=device
    )

    X_test_tensor = to_torch_tensor(
        X_test_processed,
        dtype=torch.float32,
        device=device
    )

    y_train_tensor = to_torch_tensor(
        y_train_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    y_val_tensor = to_torch_tensor(
        y_val_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    y_test_tensor = to_torch_tensor(
        y_test_processed.to_numpy().reshape(-1, 1),
        dtype=torch.float32,
        device=device
    )

    return {
        "X_train_raw": X_train,
        "X_val_raw": X_val,
        "X_test_raw": X_test,

        "y_train_raw": y_train_original,
        "y_val_raw": y_val_original,
        "y_test_raw": y_test_original,

        "X_train_processed": X_train_processed,
        "X_val_processed": X_val_processed,
        "X_test_processed": X_test_processed,

        "y_train_processed": y_train_processed,
        "y_val_processed": y_val_processed,
        "y_test_processed": y_test_processed,

        "X_train_tensor": X_train_tensor,
        "X_val_tensor": X_val_tensor,
        "X_test_tensor": X_test_tensor,

        "y_train_tensor": y_train_tensor,
        "y_val_tensor": y_val_tensor,
        "y_test_tensor": y_test_tensor,

        "preprocessor": preprocessor,
        "y_scaler": y_scaler,

        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "fingerprint_cols": fp_cols,
        "feature_names": feature_names,
    }

In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = preprocess_rt_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    target_col="rt",
    id_cols=("id", "molecule_id"),
    categorical_cols=("column.name", "column.usp.code"),
    fingerprint_col="fingerprint",
    scale_target=True,
    device=device
)

print("Preprocesamiento completado.")
print("X_train:", data["X_train_tensor"].shape)
print("X_val:", data["X_val_tensor"].shape)
print("X_test:", data["X_test_tensor"].shape)

print("y_train:", data["y_train_tensor"].shape)
print("y_val:", data["y_val_tensor"].shape)
print("y_test:", data["y_test_tensor"].shape)

Preprocesamiento completado.
X_train: torch.Size([149043, 2233])
X_val: torch.Size([31938, 2233])
X_test: torch.Size([31938, 2233])
y_train: torch.Size([149043, 1])
y_val: torch.Size([31938, 1])
y_test: torch.Size([31938, 1])


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.2 MB/s eta 0:00:00


In [ ]:
# ============================================================
# OPTUNA GLOBAL MODEL
# Full resumable version
# Saves Optuna DB, final global model, global metrics,
# by-id metrics, predictions and real values in Google Drive
# ============================================================

import os
import glob
import copy
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna

from google.colab import drive
from optuna.trial import TrialState
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


drive.mount("/content/drive")

output_dir_global = "/content/drive/MyDrive/TFG_Optuna/Global_Optuna"
os.makedirs(output_dir_global, exist_ok=True)

print("Global results will be saved in:")
print(output_dir_global)


# If an old local global Optuna DB exists, copy it to Drive once
local_global_db_files = glob.glob("optuna_global_study.db*")

for local_path in local_global_db_files:
    drive_path = os.path.join(output_dir_global, os.path.basename(local_path))

    if not os.path.exists(drive_path):
        shutil.copy2(local_path, drive_path)
        print(f"Copied local global DB file to Drive: {drive_path}")
    else:
        print(f"Drive global DB file already exists, not overwritten: {drive_path}")


#  GENERAL UTILITIES


def ensure_directory(path):
    os.makedirs(path, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_torch_tensor(x, dtype=torch.float32, device=None):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.to_numpy()

    x = np.asarray(x)
    tensor = torch.tensor(x, dtype=dtype)

    if device is not None:
        tensor = tensor.to(device)

    return tensor


def to_numpy_vector(x, dtype=np.float32):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.to_numpy()

    return np.asarray(x, dtype=dtype).reshape(-1)


def to_numpy_column(x, dtype=np.float32):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        x = x.to_numpy()

    return np.asarray(x, dtype=dtype).reshape(-1, 1)


def inverse_transform_target(y_scaled, y_scaler):
    if isinstance(y_scaled, torch.Tensor):
        y_scaled = y_scaled.detach().cpu().numpy()

    if isinstance(y_scaled, pd.Series):
        y_scaled = y_scaled.to_numpy()

    if isinstance(y_scaled, pd.DataFrame):
        y_scaled = y_scaled.to_numpy()

    y_scaled = np.asarray(y_scaled, dtype=np.float32).reshape(-1, 1)

    if y_scaler is None:
        return y_scaled.reshape(-1)

    return y_scaler.inverse_transform(y_scaled).reshape(-1)


def mean_relative_error(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom))


def vector_to_comma_string(arr, decimals=8):
    arr = np.asarray(arr, dtype=np.float64).reshape(-1)
    return ",".join(f"{x:.{decimals}f}" for x in arr)


def curve_to_comma_string(values, decimals=8):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return ",".join(f"{x:.{decimals}f}" for x in values)


def append_row_to_tsv(row, output_tsv_path):
    row_df = pd.DataFrame([row])
    write_header = not os.path.exists(output_tsv_path)

    row_df.to_csv(
        output_tsv_path,
        sep="\t",
        index=False,
        mode="a",
        header=write_header
    )

    try:
        os.sync()
    except Exception:
        pass


def load_finished_ids_from_results_tsv(results_tsv_path, id_col="id"):
    if not os.path.exists(results_tsv_path):
        return set()

    df = pd.read_csv(results_tsv_path, sep="\t")

    if id_col not in df.columns:
        return set()

    return set(df[id_col].dropna().astype(str).tolist())


def get_study_trial_counts(study):
    n_complete = sum(t.state == TrialState.COMPLETE for t in study.trials)
    n_pruned = sum(t.state == TrialState.PRUNED for t in study.trials)
    n_failed = sum(t.state == TrialState.FAIL for t in study.trials)
    n_running = sum(t.state == TrialState.RUNNING for t in study.trials)

    n_finished = n_complete + n_pruned + n_failed

    return {
        "n_trials_total": len(study.trials),
        "n_trials_complete": n_complete,
        "n_trials_pruned": n_pruned,
        "n_trials_failed": n_failed,
        "n_trials_running": n_running,
        "n_trials_finished": n_finished,
    }


def get_valid_ids_for_individual_optimization(
    train_df,
    val_df,
    test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30
):
    train_counts = train_df[id_col].value_counts()
    val_counts = val_df[id_col].value_counts()
    test_counts = test_df[id_col].value_counts()

    candidate_ids = sorted(
        set(train_counts.index) &
        set(val_counts.index) &
        set(test_counts.index)
    )

    valid_ids = []
    invalid_rows = []

    for selected_id in candidate_ids:
        n_train = int(train_counts.get(selected_id, 0))
        n_val = int(val_counts.get(selected_id, 0))
        n_test = int(test_counts.get(selected_id, 0))

        if n_train >= min_train and n_val >= min_val and n_test >= min_test:
            valid_ids.append(selected_id)
        else:
            invalid_rows.append({
                "id": selected_id,
                "n_train": n_train,
                "n_val": n_val,
                "n_test": n_test,
                "error": f"Insufficient rows: train={n_train}, val={n_val}, test={n_test}"
            })

    invalid_ids_df = pd.DataFrame(invalid_rows)

    return valid_ids, invalid_ids_df


#  MODEL


class RetentionTimeNNOptuna(nn.Module):
    def __init__(self, input_dim: int, hidden_dim_1: int, hidden_dim_2: int, dropout: float):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, 1)
        )

    def forward(self, x):
        return self.network(x)


#  GLOBAL DATA EXTRACTION


def get_global_all_features_data(prepared_data):
    X_train = prepared_data["X_train_processed"].astype(np.float32)
    X_val = prepared_data["X_val_processed"].astype(np.float32)
    X_test = prepared_data["X_test_processed"].astype(np.float32)

    y_train_processed = prepared_data["y_train_processed"].astype(np.float32)
    y_val_processed = prepared_data["y_val_processed"].astype(np.float32)
    y_test_processed = prepared_data["y_test_processed"].astype(np.float32)

    y_train_raw = to_numpy_vector(prepared_data["y_train_raw"], dtype=np.float32)
    y_val_raw = to_numpy_vector(prepared_data["y_val_raw"], dtype=np.float32)
    y_test_raw = to_numpy_vector(prepared_data["y_test_raw"], dtype=np.float32)

    return {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,

        "y_train_processed": y_train_processed,
        "y_val_processed": y_val_processed,
        "y_test_processed": y_test_processed,

        "y_train_raw": y_train_raw,
        "y_val_raw": y_val_raw,
        "y_test_raw": y_test_raw,

        "n_train_global": len(X_train),
        "n_val_global": len(X_val),
        "n_test_global": len(X_test),
        "n_features": X_train.shape[1],
        "feature_cols": list(X_train.columns),
    }



# ONE GLOBAL OPTUNA TRIAL

def train_one_trial_global(
    trial,
    global_data,
    y_scaler,
    device,
    seed=42,
    patience=20,
    min_delta=1e-4
):
    set_seed(seed)

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.35)

    hidden_dim_pair = trial.suggest_categorical(
        "hidden_dim_pair",
        [
            "128_64",
            "128_128",
            "256_64",
            "256_128",
            "256_256",
            "512_64",
            "512_128",
            "512_256",
            "512_512",
            "1024_64",
            "1024_128",
            "1024_256",
            "1024_512",
        ]
    )

    hidden_dim_1, hidden_dim_2 = map(int, hidden_dim_pair.split("_"))

    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    max_epochs = trial.suggest_int("max_epochs", 80, 300)

    X_train_t = to_torch_tensor(global_data["X_train"], dtype=torch.float32, device=None)
    X_val_t = to_torch_tensor(global_data["X_val"], dtype=torch.float32, device=device)

    y_train_t = to_torch_tensor(
        to_numpy_column(global_data["y_train_processed"], dtype=np.float32),
        dtype=torch.float32,
        device=None
    )

    y_val_raw = global_data["y_val_raw"]

    model = RetentionTimeNNOptuna(
        input_dim=X_train_t.shape[1],
        hidden_dim_1=hidden_dim_1,
        hidden_dim_2=hidden_dim_2,
        dropout=dropout
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    train_dataset = TensorDataset(X_train_t, y_train_t)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    best_val_mae = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    train_losses = []
    val_maes = []

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred_batch = model(X_batch)
            train_loss = criterion(y_pred_batch, y_batch)

            train_loss.backward()
            optimizer.step()

            batch_losses.append(train_loss.item())

        mean_train_loss = float(np.mean(batch_losses))
        train_losses.append(mean_train_loss)

        model.eval()
        with torch.no_grad():
            y_val_pred_scaled = model(X_val_t)

        y_val_pred = inverse_transform_target(y_val_pred_scaled, y_scaler)
        val_mae = mean_absolute_error(y_val_raw, y_val_pred)

        val_maes.append(float(val_mae))

        trial.report(val_mae, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_mae < best_val_mae - min_delta:
            best_val_mae = float(val_mae)
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return {
        "best_val_mae": best_val_mae,
        "best_model_state": best_model_state,
        "train_losses": train_losses,
        "val_maes": val_maes,
        "trained_epochs": len(train_losses),
        "params": {
            "learning_rate": learning_rate,
            "dropout": dropout,
            "hidden_dim_pair": hidden_dim_pair,
            "hidden_dim_1": hidden_dim_1,
            "hidden_dim_2": hidden_dim_2,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "max_epochs": max_epochs,
        }
    }



#  RETRAIN BEST GLOBAL MODEL


def retrain_best_global_model(
    best_params,
    global_data,
    y_scaler,
    device,
    seed=42,
    patience=20,
    min_delta=1e-4
):
    set_seed(seed)

    X_train_t = to_torch_tensor(global_data["X_train"], dtype=torch.float32, device=None)
    X_val_t = to_torch_tensor(global_data["X_val"], dtype=torch.float32, device=device)

    y_train_t = to_torch_tensor(
        to_numpy_column(global_data["y_train_processed"], dtype=np.float32),
        dtype=torch.float32,
        device=None
    )

    y_val_raw = global_data["y_val_raw"]

    hidden_dim_1, hidden_dim_2 = map(int, best_params["hidden_dim_pair"].split("_"))

    model = RetentionTimeNNOptuna(
        input_dim=X_train_t.shape[1],
        hidden_dim_1=hidden_dim_1,
        hidden_dim_2=hidden_dim_2,
        dropout=float(best_params["dropout"])
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(best_params["learning_rate"]),
        weight_decay=float(best_params["weight_decay"])
    )

    batch_size = int(best_params["batch_size"])
    max_epochs = int(best_params["max_epochs"])

    train_dataset = TensorDataset(X_train_t, y_train_t)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    best_val_mae = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    train_losses = []
    val_maes = []

    for epoch in range(max_epochs):
        model.train()
        batch_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred_batch = model(X_batch)
            train_loss = criterion(y_pred_batch, y_batch)

            train_loss.backward()
            optimizer.step()

            batch_losses.append(train_loss.item())

        mean_train_loss = float(np.mean(batch_losses))
        train_losses.append(mean_train_loss)

        model.eval()
        with torch.no_grad():
            y_val_pred_scaled = model(X_val_t)

        y_val_pred = inverse_transform_target(y_val_pred_scaled, y_scaler)
        val_mae = mean_absolute_error(y_val_raw, y_val_pred)

        val_maes.append(float(val_mae))

        if val_mae < best_val_mae - min_delta:
            best_val_mae = float(val_mae)
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return {
        "model": model,
        "best_val_mae": best_val_mae,
        "last_train_loss": float(train_losses[-1]),
        "last_val_mae": float(val_maes[-1]),
        "train_losses": train_losses,
        "val_maes": val_maes,
        "trained_epochs": len(train_losses),
    }



#  SAVE GLOBAL MODEL AND TRIALS


def export_global_study_trials_to_tsv(study, trials_tsv_path):
    trials_df = study.trials_dataframe()
    trials_df.insert(0, "study_name", study.study_name)

    trials_df.to_csv(trials_tsv_path, sep="\t", index=False)

    try:
        os.sync()
    except Exception:
        pass

    return trials_tsv_path


def save_global_model_checkpoint(
    model,
    global_summary,
    best_params,
    global_data,
    train_losses,
    val_maes,
    model_path
):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "global_summary": global_summary,
        "best_params": best_params,
        "feature_cols": global_data["feature_cols"],
        "train_losses": train_losses,
        "val_maes": val_maes,
    }

    torch.save(checkpoint, model_path)

    try:
        os.sync()
    except Exception:
        pass

    return model_path



#  OPTIMIZE GLOBAL MODEL, RESUMABLE


def optimize_global_model_with_optuna_resumable(
    prepared_data,
    n_trials_target=80,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=None,
    storage_path="sqlite:///optuna_global_study.db",
    study_name="global_all_features_optuna",
    output_dir="/content/drive/MyDrive/TFG_Optuna/Global_Optuna",
    global_summary_tsv_path=None,
    global_trials_tsv_path=None,
    global_model_path=None,
    verbose=True
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ensure_directory(output_dir)

    global_data = get_global_all_features_data(prepared_data)
    y_scaler = prepared_data["y_scaler"]

    sampler = optuna.samplers.TPESampler(seed=seed)

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=20,
        interval_steps=5
    )

    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
        storage=storage_path,
        load_if_exists=True
    )

    counts_before = get_study_trial_counts(study)
    n_remaining_trials = max(
        n_trials_target - counts_before["n_trials_finished"],
        0
    )

    if verbose:
        print("Global Optuna study loaded.")
        print(f"Finished trials before: {counts_before['n_trials_finished']}")
        print(f"Remaining trials to run: {n_remaining_trials}")

    def objective(trial):
        result = train_one_trial_global(
            trial=trial,
            global_data=global_data,
            y_scaler=y_scaler,
            device=device,
            seed=seed,
            patience=patience,
            min_delta=min_delta
        )

        return result["best_val_mae"]

    if n_remaining_trials > 0:
        study.optimize(
            objective,
            n_trials=n_remaining_trials,
            show_progress_bar=False
        )

    counts_after = get_study_trial_counts(study)

    complete_trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE
    ]

    if len(complete_trials) == 0:
        raise ValueError("Global study has no completed Optuna trials.")

    best_params = study.best_trial.params

    final_result = retrain_best_global_model(
        best_params=best_params,
        global_data=global_data,
        y_scaler=y_scaler,
        device=device,
        seed=seed,
        patience=patience,
        min_delta=min_delta
    )

    hidden_dim_1, hidden_dim_2 = map(
        int,
        best_params["hidden_dim_pair"].split("_")
    )

    global_summary = {
        "n_train_global": int(global_data["n_train_global"]),
        "n_val_global": int(global_data["n_val_global"]),
        "n_test_global_total": int(global_data["n_test_global"]),
        "n_features": int(global_data["n_features"]),
        "feature_block": "all_features_global_train_optuna",

        "learning_rate": float(best_params["learning_rate"]),
        "epochs": int(final_result["trained_epochs"]),

        "best_val_loss": float(final_result["best_val_mae"]),
        "last_train_loss": float(final_result["last_train_loss"]),
        "last_val_loss": float(final_result["last_val_mae"]),

        "optuna_trials_target": int(n_trials_target),
        "optuna_trials_total": int(counts_after["n_trials_total"]),
        "optuna_trials_finished": int(counts_after["n_trials_finished"]),
        "optuna_trials_complete": int(counts_after["n_trials_complete"]),
        "optuna_trials_pruned": int(counts_after["n_trials_pruned"]),
        "optuna_trials_failed": int(counts_after["n_trials_failed"]),
        "optuna_trials_running": int(counts_after["n_trials_running"]),

        "best_trial_number": int(study.best_trial.number),
        "best_value_val_MAE": float(study.best_value),
        "trained_epochs_final_model": int(final_result["trained_epochs"]),

        "best_learning_rate": float(best_params["learning_rate"]),
        "best_dropout": float(best_params["dropout"]),
        "best_hidden_dim_pair": best_params["hidden_dim_pair"],
        "best_hidden_dim_1": int(hidden_dim_1),
        "best_hidden_dim_2": int(hidden_dim_2),
        "best_weight_decay": float(best_params["weight_decay"]),
        "best_batch_size": int(best_params["batch_size"]),
        "best_max_epochs": int(best_params["max_epochs"]),

        "best_val_mae": float(final_result["best_val_mae"]),
        "last_val_mae": float(final_result["last_val_mae"]),

        "train_loss_curve": curve_to_comma_string(final_result["train_losses"]),
        "val_mae_curve": curve_to_comma_string(final_result["val_maes"]),

        "study_name": study.study_name,
        "storage_path": storage_path,
    }

    if global_summary_tsv_path is not None:
        pd.DataFrame([global_summary]).to_csv(
            global_summary_tsv_path,
            sep="\t",
            index=False
        )

    if global_trials_tsv_path is not None:
        export_global_study_trials_to_tsv(
            study=study,
            trials_tsv_path=global_trials_tsv_path
        )

    if global_model_path is not None:
        save_global_model_checkpoint(
            model=final_result["model"],
            global_summary=global_summary,
            best_params=best_params,
            global_data=global_data,
            train_losses=final_result["train_losses"],
            val_maes=final_result["val_maes"],
            model_path=global_model_path
        )

        global_summary["model_path"] = global_model_path
    else:
        global_summary["model_path"] = None

    if verbose:
        print("Global Optuna finished.")
        print(f"Best validation MAE: {study.best_value:.4f}")
        print(f"Finished trials: {counts_after['n_trials_finished']}")
        print(f"Best params: {best_params}")

    return {
        "model": final_result["model"],
        "study": study,
        "global_summary": global_summary,
        "global_data": global_data,
        "train_losses": final_result["train_losses"],
        "val_maes": final_result["val_maes"],
    }


#  EVALUATE GLOBAL MODEL ON FULL TEST


def evaluate_global_model_on_full_test(
    model,
    global_data,
    y_scaler,
    device=None,
    predictions_tsv_path=None,
    summary_tsv_path=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_test_t = to_torch_tensor(global_data["X_test"], dtype=torch.float32, device=device)

    y_test_t = to_torch_tensor(
        to_numpy_column(global_data["y_test_processed"], dtype=np.float32),
        dtype=torch.float32,
        device=device
    )

    model.eval()
    with torch.no_grad():
        y_test_pred_scaled = model(X_test_t)

    y_test_pred = inverse_transform_target(y_test_pred_scaled, y_scaler)
    y_test_real = inverse_transform_target(y_test_t, y_scaler)

    mae = float(mean_absolute_error(y_test_real, y_test_pred))
    rmse = float(np.sqrt(mean_squared_error(y_test_real, y_test_pred)))
    r2 = float(r2_score(y_test_real, y_test_pred))
    mre = float(mean_relative_error(y_test_real, y_test_pred))

    summary = {
        "scope": "full_test",
        "n_test": int(len(y_test_real)),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MRE": mre,
    }

    if summary_tsv_path is not None:
        pd.DataFrame([summary]).to_csv(
            summary_tsv_path,
            sep="\t",
            index=False
        )

    if predictions_tsv_path is not None:
        predictions_df = pd.DataFrame({
            "row_position": np.arange(len(y_test_real)),
            "y_test_real": y_test_real,
            "y_test_pred": y_test_pred,
            "absolute_error": np.abs(y_test_real - y_test_pred)
        })

        predictions_df.to_csv(
            predictions_tsv_path,
            sep="\t",
            index=False
        )

    try:
        os.sync()
    except Exception:
        pass

    return summary


#  EVALUATE GLOBAL MODEL BY ID, RESUMABLE


def evaluate_global_optuna_model_by_id_resumable(
    model,
    prepared_data,
    train_df,
    val_df,
    test_df,
    valid_ids,
    global_summary,
    id_col="id",
    device=None,
    results_tsv_path="optuna_global_results_by_id.tsv",
    failed_ids_tsv_path="optuna_global_failed_ids.tsv",
    decimals=8,
    verbose=True
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    y_scaler = prepared_data["y_scaler"]

    finished_ids = load_finished_ids_from_results_tsv(
        results_tsv_path=results_tsv_path,
        id_col="id"
    )

    if verbose:
        print(f"Global by-id evaluation.")
        print(f"Already evaluated ids: {len(finished_ids)}")

    for selected_id in valid_ids:
        selected_id_str = str(selected_id)

        if selected_id_str in finished_ids:
            if verbose:
                print(f"Skipping ID {selected_id}, already saved in global results TSV")
            continue

        try:
            train_idx = train_df.loc[train_df[id_col] == selected_id].index
            val_idx = val_df.loc[val_df[id_col] == selected_id].index
            test_idx = test_df.loc[test_df[id_col] == selected_id].index

            X_test_id = prepared_data["X_test_processed"].loc[test_idx].astype(np.float32)
            y_test_processed_id = prepared_data["y_test_processed"].loc[test_idx].astype(np.float32)

            X_test_t = to_torch_tensor(X_test_id, dtype=torch.float32, device=device)

            y_test_t = to_torch_tensor(
                to_numpy_column(y_test_processed_id, dtype=np.float32),
                dtype=torch.float32,
                device=device
            )

            model.eval()
            with torch.no_grad():
                y_test_pred_scaled = model(X_test_t)

            y_test_pred = inverse_transform_target(y_test_pred_scaled, y_scaler)
            y_test_real = inverse_transform_target(y_test_t, y_scaler)

            mae = float(mean_absolute_error(y_test_real, y_test_pred))
            rmse = float(np.sqrt(mean_squared_error(y_test_real, y_test_pred)))
            r2 = float(r2_score(y_test_real, y_test_pred))
            mre = float(mean_relative_error(y_test_real, y_test_pred))

            row = {
                "id": selected_id,

                "n_train_id": int(len(train_idx)),
                "n_val_id": int(len(val_idx)),
                "n_test_id": int(len(test_idx)),

                "n_train_global": global_summary["n_train_global"],
                "n_val_global": global_summary["n_val_global"],
                "n_test_global_total": global_summary["n_test_global_total"],
                "n_features": global_summary["n_features"],
                "feature_block": global_summary["feature_block"],

                "learning_rate": global_summary["learning_rate"],
                "epochs": global_summary["epochs"],

                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "MRE": mre,

                "best_val_loss": global_summary["best_val_loss"],
                "last_train_loss": global_summary["last_train_loss"],
                "last_val_loss": global_summary["last_val_loss"],

                "y_test_real_vector": vector_to_comma_string(
                    y_test_real,
                    decimals=decimals
                ),
                "y_test_pred_vector": vector_to_comma_string(
                    y_test_pred,
                    decimals=decimals
                ),

                "optuna_trials_target": global_summary["optuna_trials_target"],
                "optuna_trials_total": global_summary["optuna_trials_total"],
                "optuna_trials_finished": global_summary["optuna_trials_finished"],
                "optuna_trials_complete": global_summary["optuna_trials_complete"],
                "optuna_trials_pruned": global_summary["optuna_trials_pruned"],
                "optuna_trials_failed": global_summary["optuna_trials_failed"],
                "optuna_trials_running": global_summary["optuna_trials_running"],

                "best_trial_number": global_summary["best_trial_number"],
                "best_value_val_MAE": global_summary["best_value_val_MAE"],
                "trained_epochs_final_model": global_summary["trained_epochs_final_model"],

                "best_learning_rate": global_summary["best_learning_rate"],
                "best_dropout": global_summary["best_dropout"],
                "best_hidden_dim_pair": global_summary["best_hidden_dim_pair"],
                "best_hidden_dim_1": global_summary["best_hidden_dim_1"],
                "best_hidden_dim_2": global_summary["best_hidden_dim_2"],
                "best_weight_decay": global_summary["best_weight_decay"],
                "best_batch_size": global_summary["best_batch_size"],
                "best_max_epochs": global_summary["best_max_epochs"],

                "best_val_mae": global_summary["best_val_mae"],
                "last_val_mae": global_summary["last_val_mae"],

                "study_name": global_summary["study_name"],
                "storage_path": global_summary["storage_path"],
                "model_path": global_summary["model_path"],
            }

            append_row_to_tsv(
                row=row,
                output_tsv_path=results_tsv_path
            )

            finished_ids.add(selected_id_str)

            if verbose:
                print(f"ID {selected_id} global evaluation saved. MAE: {mae:.4f}")

        except Exception as e:
            failed_row = {
                "id": selected_id,
                "error": str(e)
            }

            append_row_to_tsv(
                row=failed_row,
                output_tsv_path=failed_ids_tsv_path
            )

            if verbose:
                print(f"ID {selected_id} failed in global evaluation: {e}")

    if os.path.exists(results_tsv_path):
        results_df = pd.read_csv(results_tsv_path, sep="\t")
    else:
        results_df = pd.DataFrame()

    if os.path.exists(failed_ids_tsv_path):
        failed_ids_df = pd.read_csv(failed_ids_tsv_path, sep="\t")
    else:
        failed_ids_df = pd.DataFrame()

    return results_df, failed_ids_df



#  RUN GLOBAL OPTUNA AND EVALUATION, RESUMABLE


def run_optuna_global_and_evaluate_valid_ids_resumable(
    prepared_data,
    train_df,
    val_df,
    test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30,
    n_trials_target=80,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=None,
    output_dir="/content/drive/MyDrive/TFG_Optuna/Global_Optuna",
    storage_filename="optuna_global_study.db",
    study_name="global_all_features_optuna",
    results_tsv_filename="optuna_global_results_by_id.tsv",
    failed_ids_tsv_filename="optuna_global_failed_ids.tsv",
    invalid_ids_tsv_filename="optuna_global_invalid_ids.tsv",
    global_summary_tsv_filename="optuna_global_summary.tsv",
    global_trials_tsv_filename="optuna_global_trials.tsv",
    global_model_filename="optuna_global_model.pt",
    full_test_summary_tsv_filename="optuna_global_full_test_summary.tsv",
    full_test_predictions_tsv_filename="optuna_global_full_test_predictions.tsv",
    decimals=8,
    verbose=True
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ensure_directory(output_dir)

    storage_db_path = os.path.join(output_dir, storage_filename)
    storage_path = f"sqlite:///{storage_db_path}"

    results_tsv_path = os.path.join(output_dir, results_tsv_filename)
    failed_ids_tsv_path = os.path.join(output_dir, failed_ids_tsv_filename)
    invalid_ids_tsv_path = os.path.join(output_dir, invalid_ids_tsv_filename)

    global_summary_tsv_path = os.path.join(output_dir, global_summary_tsv_filename)
    global_trials_tsv_path = os.path.join(output_dir, global_trials_tsv_filename)
    global_model_path = os.path.join(output_dir, global_model_filename)

    full_test_summary_tsv_path = os.path.join(output_dir, full_test_summary_tsv_filename)
    full_test_predictions_tsv_path = os.path.join(output_dir, full_test_predictions_tsv_filename)

    valid_ids, invalid_ids_df = get_valid_ids_for_individual_optimization(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        min_train=min_train,
        min_val=min_val,
        min_test=min_test
    )

    if len(invalid_ids_df) > 0:
        invalid_ids_df.to_csv(
            invalid_ids_tsv_path,
            sep="\t",
            index=False
        )

    if verbose:
        print(f"Valid ids for global evaluation: {len(valid_ids)}")
        print(f"Using device: {device}")
        print(f"Output directory: {output_dir}")
        print(f"Optuna DB: {storage_db_path}")
        print(f"Global by-id results TSV: {results_tsv_path}")

    global_result = optimize_global_model_with_optuna_resumable(
        prepared_data=prepared_data,
        n_trials_target=n_trials_target,
        seed=seed,
        patience=patience,
        min_delta=min_delta,
        device=device,
        storage_path=storage_path,
        study_name=study_name,
        output_dir=output_dir,
        global_summary_tsv_path=global_summary_tsv_path,
        global_trials_tsv_path=global_trials_tsv_path,
        global_model_path=global_model_path,
        verbose=verbose
    )

    full_test_summary = evaluate_global_model_on_full_test(
        model=global_result["model"],
        global_data=global_result["global_data"],
        y_scaler=prepared_data["y_scaler"],
        device=device,
        predictions_tsv_path=full_test_predictions_tsv_path,
        summary_tsv_path=full_test_summary_tsv_path
    )

    if verbose:
        print("Full global test metrics:")
        print(full_test_summary)

    results_df, failed_ids_df = evaluate_global_optuna_model_by_id_resumable(
        model=global_result["model"],
        prepared_data=prepared_data,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        valid_ids=valid_ids,
        global_summary=global_result["global_summary"],
        id_col=id_col,
        device=device,
        results_tsv_path=results_tsv_path,
        failed_ids_tsv_path=failed_ids_tsv_path,
        decimals=decimals,
        verbose=verbose
    )

    return {
        "results_df": results_df,
        "failed_ids_df": failed_ids_df,
        "valid_ids": valid_ids,
        "model": global_result["model"],
        "study": global_result["study"],
        "global_summary": global_result["global_summary"],
        "global_data": global_result["global_data"],
        "train_losses": global_result["train_losses"],
        "val_maes": global_result["val_maes"],
        "full_test_summary": full_test_summary,
    }



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Global results will be saved in:
/content/drive/MyDrive/TFG_Optuna/Global_Optuna


In [ ]:
global_results_path = "/content/drive/MyDrive/TFG_Optuna/Global_Optuna/optuna_global_results_by_id.tsv"

if os.path.exists(global_results_path):
    df_global_check = pd.read_csv(global_results_path, sep="\t")
    print("Filas guardadas:", len(df_global_check))
    display(df_global_check.tail())
else:
    print("Todavía no existe el TSV por id.")

Todavía no existe el TSV por id.


In [ ]:
# ============================================================
# 12. EXECUTION
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

global_optuna_result = run_optuna_global_and_evaluate_valid_ids_resumable(
    prepared_data=data,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    id_col="id",
    min_train=30,
    min_val=30,
    min_test=30,
    n_trials_target=40,
    seed=42,
    patience=20,
    min_delta=1e-4,
    device=device,
    output_dir=output_dir_global,
    storage_filename="optuna_global_study.db",
    study_name="global_all_features_optuna",
    results_tsv_filename="optuna_global_results_by_id.tsv",
    failed_ids_tsv_filename="optuna_global_failed_ids.tsv",
    invalid_ids_tsv_filename="optuna_global_invalid_ids.tsv",
    global_summary_tsv_filename="optuna_global_summary.tsv",
    global_trials_tsv_filename="optuna_global_trials.tsv",
    global_model_filename="optuna_global_model.pt",
    full_test_summary_tsv_filename="optuna_global_full_test_summary.tsv",
    full_test_predictions_tsv_filename="optuna_global_full_test_predictions.tsv",
    decimals=8,
    verbose=True
)

print("Finished global Optuna.")
print("Global by-id results saved in:")
print(os.path.join(output_dir_global, "optuna_global_results_by_id.tsv"))

Device: cuda
GPU: Tesla T4
Valid ids for global evaluation: 141
Using device: cuda
Output directory: /content/drive/MyDrive/TFG_Optuna/Global_Optuna
Optuna DB: /content/drive/MyDrive/TFG_Optuna/Global_Optuna/optuna_global_study.db
Global by-id results TSV: /content/drive/MyDrive/TFG_Optuna/Global_Optuna/optuna_global_results_by_id.tsv


[I 2026-06-24 20:29:13,348] A new study created in RDB with name: global_all_features_optuna


Global Optuna study loaded.
Finished trials before: 0
Remaining trials to run: 40


[I 2026-06-24 21:08:11,139] Trial 0 finished with value: 0.7816577553749084 and parameters: {'learning_rate': 0.00043284502212938834, 'dropout': 0.33275000724347065, 'hidden_dim_pair': '1024_64', 'weight_decay': 5.415244119402535e-07, 'batch_size': 32, 'max_epochs': 215}. Best is trial 0 with value: 0.7816577553749084.
[I 2026-06-24 21:32:11,898] Trial 1 finished with value: 0.7254913449287415 and parameters: {'learning_rate': 0.00017258215396625024, 'dropout': 0.10225062698732634, 'hidden_dim_pair': '1024_256', 'weight_decay': 1.6536937182824404e-06, 'batch_size': 32, 'max_epochs': 189}. Best is trial 1 with value: 0.7254913449287415.
[I 2026-06-24 22:08:00,270] Trial 2 finished with value: 0.7031762003898621 and parameters: {'learning_rate': 0.00011439974749291271, 'dropout': 0.3182621407275737, 'hidden_dim_pair': '512_128', 'weight_decay': 6.080390190296596e-07, 'batch_size': 64, 'max_epochs': 263}. Best is trial 2 with value: 0.7031762003898621.
[I 2026-06-24 22:35:17,835] Trial 3 

Global Optuna finished.
Best validation MAE: 0.6866
Finished trials: 40
Best params: {'learning_rate': 0.00010522453501874268, 'dropout': 0.20123477237404616, 'hidden_dim_pair': '1024_256', 'weight_decay': 3.5131141921369886e-06, 'batch_size': 128, 'max_epochs': 247}
Full global test metrics:
{'scope': 'full_test', 'n_test': 31938, 'MAE': 0.6840353012084961, 'RMSE': 1.3584436756008578, 'R2': 0.949406623840332, 'MRE': 39984.25390625}
Global by-id evaluation.
Already evaluated ids: 0
ID 2 global evaluation saved. MAE: 2.6626
ID 4 global evaluation saved. MAE: 0.5894
ID 7 global evaluation saved. MAE: 0.9263
ID 9 global evaluation saved. MAE: 0.2523
ID 19 global evaluation saved. MAE: 0.7213
ID 27 global evaluation saved. MAE: 0.7760
ID 44 global evaluation saved. MAE: 0.9960
ID 45 global evaluation saved. MAE: 0.6677
ID 47 global evaluation saved. MAE: 5.6173
ID 48 global evaluation saved. MAE: 0.4246
ID 54 global evaluation saved. MAE: 0.2855
ID 55 global evaluation saved. MAE: 0.3205
I